# Exercise 4: Regression, Regularization, and Baselines

Exercise 3 prepared a table so that a model can learn from it. This exercise builds the models. **Supervised learning** learns from examples whose answer is known: each row has predictors and a response, and a model learns how the response depends on the predictors so that it can predict the response of a row it has never seen. **Unsupervised learning**, which a later exercise covers, has no response and looks for structure in the predictors alone.

The value to predict is the **response** $y$, also called the target or the label, and the columns used to predict it are the **predictors** $X$, also called features. The response depends on the predictors through an unknown function $f$, plus noise $\varepsilon$ that no predictor explains:

$$y = f(X) + \varepsilon$$

A **model** is an estimate $\hat{f}$ of that function, and a prediction is $\hat{y} = \hat{f}(x)$ for a new row $x$. The task is called:

- **regression** when the response is a number, such as the value of a house, and
- **classification** when the response is a category, such as whether a tumour is malignant.

Every learning method has three parts: a **representation**, the family of models it searches, such as straight lines; an **evaluation**, the score that ranks the models, such as the mean squared error; and an **optimization**, the search for the best model, such as a formula or gradient descent. A model has **parameters**, which it learns from the data, such as the slope of a line, and **hyperparameters**, which the analyst sets before learning, such as the degree of a polynomial.

A model is useful only if it predicts rows it has never seen, so every model is scored on held-out rows and compared with a **baseline**, a model that ignores the predictors.

A small example fixes the words. A bank wants to estimate the value of a house before it lends money. It has a table of past sales: each row is one house, the columns describe it (its size, its age, the income of its neighbourhood), and one column holds the price it sold for. The descriptive columns are the predictors, the price is the response, and every past sale is a **training example** whose answer is known. A model learned from these rows can then price a house that has not been sold yet. Section 1 shows such rows for California.

The exercise predicts the median house value of 20,640 districts of California in 1990, from `data/housing.csv`. It starts from a baseline that ignores the predictors, fits straight lines by a formula and by gradient descent, adds polynomial features until they overfit, and tames them with penalties chosen by cross-validation. Exercise 5 turns to classification, where the response is a category, and Exercise 6 to trees and their ensembles.

## Setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, Lasso, LassoCV, LinearRegression, Ridge, RidgeCV, lasso_path
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error, mean_squared_error,
                             r2_score, root_mean_squared_error)
from sklearn.model_selection import KFold, cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

DATA = Path("data")
SEED = 2026
rng = np.random.default_rng(SEED)
# The lasso stops at max_iter on some penalties, and its result there is still usable.
warnings.filterwarnings("ignore", message="Objective did not converge")

In [ ]:
# The course style. The colours repeat template/themes/data-science.css, so a change there is repeated here.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.facecolor": GROUND,
    "figure.constrained_layout.use": True,
    "savefig.facecolor": GROUND,
    "savefig.transparent": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Archivo", "DejaVu Sans"],
    "font.size": 12,
    "text.color": INK,
    "axes.facecolor": GROUND,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "grid.color": GRID,
    "xtick.color": INK,
    "ytick.color": INK,
    "legend.frameon": False,
    "image.cmap": "cividis",
})
sns.set_palette(PALETTE)

## 1. The Housing Data

Each row of `data/housing.csv` is one census block group, a district of a few hundred to a few thousand people, from the 1990 census of California. The file holds totals per district, such as the number of rooms, so the predictors are built as averages per household, the form that scikit-learn and many textbooks use. The response `MedHouseVal` is the median house value of the district in units of 100,000 dollars.

In [ ]:
housing = pd.read_csv(DATA / "housing.csv")
houses = pd.DataFrame({
    "MedInc": housing["median_income"],
    "HouseAge": housing["housing_median_age"],
    "AveRooms": housing["total_rooms"] / housing["households"],
    "AveBedrms": housing["total_bedrooms"] / housing["households"],
    "Population": housing["population"],
    "AveOccup": housing["population"] / housing["households"],
    "Latitude": housing["latitude"],
    "Longitude": housing["longitude"],
    "ocean_proximity": housing["ocean_proximity"],
    "MedHouseVal": housing["median_house_value"] / 100_000,
})
houses.shape

Each row is one **example**. The first three rows show what a model sees: everything except the last column is the input $x$ of a row, and `MedHouseVal` is the answer $y$ that the model must learn to predict.

In [ ]:
houses.head(3)

The predictors are:

- `MedInc`, the median income of the district, in tens of thousands of dollars,
- `HouseAge`, the median age of its houses, in years,
- `AveRooms`, `AveBedrms`, and `AveOccup`, the average numbers of rooms, bedrooms, and occupants per household,
- `Population`, the number of people in the district,
- `Latitude` and `Longitude`, the position of the district, and
- `ocean_proximity`, a label that says roughly how close the district lies to the ocean.

In [ ]:
houses.describe().T.round(2)

In [ ]:
houses.isna().sum()[lambda counts: counts > 0], houses["ocean_proximity"].value_counts()

`AveBedrms` has 207 gaps, which the preparation of Exercise 3 fills, and `ocean_proximity` is a label, which it one-hot encodes. Before any model, the response and its relationships are drawn, as Exercise 1 did for every new table.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].hist(houses["MedHouseVal"], bins=50, color=PALETTE[0])
axes[0].set(title="Median house value", xlabel="100,000 dollars", yticks=[])
axes[1].scatter(houses["MedInc"], houses["MedHouseVal"], s=2, alpha=0.3, color=PALETTE[0])
axes[1].set(title="Value against income", xlabel="Median income", ylabel="Median house value")
correlations = houses.drop(columns="ocean_proximity").corr()["MedHouseVal"]
correlations = correlations.drop("MedHouseVal").sort_values()
axes[2].barh(correlations.index, correlations, color=PALETTE[2])
axes[2].set(title="Correlation with the value", xlabel="Pearson r")
plt.show()

The values pile up at 5.0, because every value above 500,000 dollars is recorded as 500,001: the response is **capped**, and no model can learn what lies above the cap. The median income has by far the strongest correlation with the value, a correlation of about 0.69.

## 2. Training, Validation, and Test Sets

A model is judged by how well it predicts rows it has never seen. The rows are therefore split into three sets with three roles:

- the **training set** fits the parameters of every candidate model,
- the **validation set** compares the candidates and chooses their hyperparameters, and
- the **test set** plays the role of future rows: it takes no part in any decision and scores the chosen model once, at the very end.

An analogy helps. The training set is the textbook a student learns from, the validation set is the practice exams the student uses to decide how to study, and the test set is the final exam. A student who has seen the final exam in advance gets a high grade that says nothing about what they learned. In the same way, a test score used to choose between models would no longer measure performance on new data, because the choice itself would be fitted to the test rows.

Here the rows are split 60/20/20: `train_test_split` first holds out a fifth as the test set, and then a quarter of the rest as the validation set. `random_state` fixes the shuffle, so a rerun gives the same split.

In [ ]:
X, y = houses.drop(columns="MedHouseVal"), houses["MedHouseVal"]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=0.25, random_state=SEED)
len(X_train), len(X_valid), len(X_test)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.6))
sizes = {"training, 60 %": len(X_train), "validation, 20 %": len(X_valid), "test, 20 %": len(X_test)}
left = 0
for (name, size), colour in zip(sizes.items(), [PALETTE[0], PALETTE[4], PALETTE[1]]):
    ax.barh(0, size, left=left, color=colour)
    ax.text(left + size / 2, 0, f"{name}\n{size:,} districts", ha="center", va="center", color=GROUND)
    left += size
ax.set(xlim=(0, left), yticks=[], xlabel="Districts, after shuffling", title="One split, three roles")
ax.grid(False)
plt.show()

The shuffle matters: the file may be ordered, for example by region, and without shuffling the test set would hold only one part of California.

### The Interface of scikit-learn

Every model in scikit-learn is an object with the same few methods, the same interface that the preparation steps of Exercise 3 use:

- the object is created with its hyperparameters, such as `Ridge(alpha=1.0)`,
- `fit(X, y)` learns the parameters from the training rows and stores them in attributes whose names end with an underscore, such as `coef_`, and
- `predict(X)` returns the predictions for any rows.

A toy example shows the three steps. Three rows with $x = 1, 2, 3$ and $y = 2, 4, 6$ follow the rule $y = 2x$ exactly. `LinearRegression` learns the slope 2 from them, stores it in `coef_`, and predicts 20 for a new row with $x = 10$.

In [ ]:
toy_X = pd.DataFrame({"x": [1, 2, 3]})
toy = LinearRegression().fit(toy_X, [2, 4, 6])
toy.coef_.round(3), toy.predict(pd.DataFrame({"x": [10]})).round(3)

Every model in this exercise is used exactly like this, whatever happens inside `fit`. A model and its preparation are joined with `make_pipeline`, so that `fit` learns every statistic of the preparation and every parameter of the model from the training rows alone. The preparation of Exercise 3 is rebuilt here for the housing columns: the gaps are filled with the training median, every number is standardized, and the label is one-hot encoded.

In [ ]:
numeric = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude",
           "Longitude"]
preparation = ColumnTransformer([
    ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("ocean", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["ocean_proximity"]),
])
preparation.fit_transform(X_train).shape

## 3. The Baseline and the Metrics

A model is worth something only if it beats a model that ignores the predictors. `DummyRegressor` is that model: it predicts the mean value of the training rows for every district.

A **metric** turns the errors $y_i - \hat{y}_i$ of $n$ predictions into one number. Five are standard:

- the **mean absolute error**, $\text{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$, the typical size of an error, in the units of the response,
- the **mean absolute percentage error**, $\text{MAPE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i| / |y_i|$, the typical error as a share of the value,
- the **mean squared error**, $\text{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$, which squares the errors, so one large error weighs more than several small ones, in squared units,
- the **root mean squared error**, $\text{RMSE} = \sqrt{\text{MSE}}$, the same quantity back in the units of the response, and
- the **coefficient of determination**, $R^2 = 1 - \sum_i (y_i - \hat{y}_i)^2 / \sum_i (y_i - \bar{y})^2$, the share of the squared deviation from the mean that the model removes. The mean scores 0, a perfect model scores 1, and a model worse than the mean scores below 0.

`regression_scores` computes all five. Before it scores the housing models, it is tried on four made-up districts whose values and predictions are easy to check by hand: the true values are 1, 2, 3, and 4, and the predictions are 1.5, 2, 2, and 6, so the errors are 0.5, 0, 1, and 2.

In [ ]:
def regression_scores(y_true: pd.Series, y_pred: np.ndarray) -> pd.Series:
    return pd.Series({"MAE": mean_absolute_error(y_true, y_pred),
                      "MAPE": mean_absolute_percentage_error(y_true, y_pred),
                      "MSE": mean_squared_error(y_true, y_pred),
                      "RMSE": root_mean_squared_error(y_true, y_pred),
                      "R2": r2_score(y_true, y_pred)})


regression_scores(pd.Series([1.0, 2.0, 3.0, 4.0]), np.array([1.5, 2.0, 2.0, 6.0])).round(3)

Each number can be checked:

- the MAE is $(0.5 + 0 + 1 + 2) / 4 = 0.875$,
- the MAPE is $(0.5/1 + 0/2 + 1/3 + 2/4) / 4 = 0.333$, a third of the value on average,
- the MSE is $(0.25 + 0 + 1 + 4) / 4 = 1.3125$: the single error of 2 contributes 4 of the 5.25, which is what "squaring punishes large errors" means,
- the RMSE is $\sqrt{1.3125} = 1.146$, larger than the MAE because of that one large error, and
- $R^2$ is negative: the mean of the true values, 2.5, would have a squared error of $1.5^2 + 0.5^2 + 0.5^2 + 1.5^2 = 5$, less than the model's 5.25, so this model is worse than predicting the mean.

`results` collects the validation scores of every candidate, starting with the baseline.

In [ ]:
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
results = pd.DataFrame({"baseline": regression_scores(y_valid, baseline.predict(X_valid))}).T
results.round(3)

The baseline misses a district by about 93,000 dollars on average, and its $R^2$ is 0. The models are compared by RMSE, because a large error on one district costs more than several small ones, and MAPE is reported beside it, because it reads the error relative to the value.

## 4. Simple Linear Regression

**Linear regression** assumes that the response changes by a fixed amount for each unit of a predictor. With one predictor, the model is a straight line:

$$\hat{y} = \beta_0 + \beta_1 x$$

The **intercept** $\beta_0$ is the prediction at $x = 0$, and the **slope** $\beta_1$ is the change in the prediction when $x$ grows by one unit. The **residual** of a row, $e_i = y_i - \hat{y}_i$, is the vertical distance from its point to the line. **Least squares** chooses the line with the smallest mean squared residual, and for one predictor, setting the derivatives of the MSE to zero gives a closed form:

$$\hat{\beta}_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

In [ ]:
x = X_train["MedInc"].to_numpy()
slope = np.sum((x - x.mean()) * (y_train - y_train.mean())) / np.sum((x - x.mean()) ** 2)
intercept = y_train.mean() - slope * x.mean()
simple = LinearRegression().fit(X_train[["MedInc"]], y_train)
(round(intercept, 3), round(slope, 3)), (round(simple.intercept_, 3), round(simple.coef_[0], 3))

The figure below draws the idea on 25 training districts: the line, each point, and the residual of each point as a vertical segment. Least squares chooses the line that makes the sum of the squared lengths of these segments as small as possible.

In [ ]:
sample = X_train.sample(25, random_state=SEED).index
income_sample, value_sample = X_train.loc[sample, "MedInc"], y_train[sample]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.vlines(income_sample, value_sample, intercept + slope * income_sample, color=PALETTE[1], linewidth=1)
ax.scatter(income_sample, value_sample, color=PALETTE[0], zorder=3)
grid = np.linspace(0, income_sample.max(), 50)
ax.plot(grid, intercept + slope * grid, color=INK)
ax.set(xlabel="Median income", ylabel="Median house value",
       title="Residuals are the vertical distances to the line")
plt.show()

`LinearRegression` finds the same line and stores it in `intercept_` and `coef_`. Each additional 10,000 dollars of median income is associated with about 42,000 dollars more in median house value. For a district with a median income of 5, that is 50,000 dollars, the line predicts $0.445 + 0.419 \times 5 = 2.54$, about 254,000 dollars. The coefficient describes an association between districts, not the effect of raising anyone's income.

In [ ]:
results.loc["simple linear"] = regression_scores(y_valid, simple.predict(X_valid[["MedInc"]]))
results.round(3)

### Gradient Descent

A closed form exists for least squares, but not for most models. **Gradient descent** finds the minimum of a loss by repeated small steps downhill. The gradient is the vector of derivatives of the loss, and it points uphill, so each step moves the coefficients against it:

$$\beta \leftarrow \beta - \eta \, \nabla L(\beta), \qquad \frac{\partial L}{\partial \beta_0} = -\frac{2}{n}\sum_i e_i, \qquad \frac{\partial L}{\partial \beta_1} = -\frac{2}{n}\sum_i x_i e_i$$

Picture standing on a hillside in fog: the ground can be felt but not seen. The steepest way down is known at every point, so a few small steps in that direction, repeated, lead to the bottom of the valley. The loss is the height, the coefficients are the position, and the gradient is the slope underfoot.

The **learning rate** $\eta$ sets the size of a step: too small wastes steps, and too large overshoots and diverges. The predictor is standardized first, so that one learning rate suits both coefficients. In the loop below, `residual` holds the errors of the current line, and each coefficient moves by $\eta$ times its derivative, with the sign that lowers the MSE.

In [ ]:
z = (x - x.mean()) / x.std()
b0, b1, eta = 0.0, 0.0, 0.1
history = []
for _ in range(60):
    residual = y_train.to_numpy() - (b0 + b1 * z)
    b0 = b0 + eta * 2 * residual.mean()
    b1 = b1 + eta * 2 * (z * residual).mean()
    history.append((residual ** 2).mean())
round(float(b0), 3), round(float(b1), 3)

In [ ]:
round(float(intercept + slope * x.mean()), 3), round(float(slope * x.std()), 3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].scatter(x, y_train, s=2, alpha=0.3, color=PALETTE[0])
grid = np.linspace(x.min(), x.max(), 50)
axes[0].plot(grid, intercept + slope * grid, color=PALETTE[1])
axes[0].set(xlabel="Median income", ylabel="Median house value", title="The least-squares line")
axes[1].plot(history, color=PALETTE[2])
axes[1].set(yscale="log", xlabel="Iteration", ylabel="Training MSE",
            title=f"Gradient descent, learning rate {eta}")
plt.show()

On the standardized predictor, the intercept is the mean value and the slope is the change per standard deviation of income, and both agree with the closed form. The training MSE falls quickly and then flattens at the minimum.

The same loop with three other learning rates shows why the rate matters: a rate of 0.01 is still far from the bottom after 60 steps, and a rate of 1.05 overshoots the valley further at every step, so the error grows instead of falling.

In [ ]:
def descend(rate: float, steps: int = 60) -> list:
    b0_now, b1_now, path = 0.0, 0.0, []
    for _ in range(steps):
        error = y_train.to_numpy() - (b0_now + b1_now * z)
        b0_now, b1_now = b0_now + rate * 2 * error.mean(), b1_now + rate * 2 * (z * error).mean()
        path.append((error ** 2).mean())
    return path


fig, ax = plt.subplots(figsize=(9, 3.4))
for rate, colour in [(0.01, PALETTE[4]), (0.1, PALETTE[2]), (1.05, PALETTE[1])]:
    ax.plot(descend(rate), color=colour, label=f"learning rate {rate}")
ax.set(yscale="log", xlabel="Iteration", ylabel="Training MSE", title="Three learning rates")
ax.legend()
plt.show()

## 5. Multiple Linear Regression

**Multiple linear regression** uses several predictors at once:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p$$

Least squares again chooses the coefficients with the smallest training MSE. The first model uses the eight numerical columns as they are, with only their gaps filled, and the second uses the full preparation, which adds the ocean proximity.

In [ ]:
fill_only = ColumnTransformer([("numeric", SimpleImputer(strategy="median"), numeric)])
raw_linear = make_pipeline(fill_only, LinearRegression()).fit(X_train, y_train)
results.loc["multiple linear, 8 columns"] = regression_scores(y_valid, raw_linear.predict(X_valid))
linear = make_pipeline(preparation, LinearRegression()).fit(X_train, y_train)
results.loc["multiple linear, prepared"] = regression_scores(y_valid, linear.predict(X_valid))
results.round(3)

The pipeline is fitted on the training rows and scored on the validation rows. Its coefficients can be read from the fitted steps: `linear[0]` is the preparation and `linear[-1]` is the model.

In [ ]:
coefficients = pd.Series(linear[-1].coef_, index=linear[0].get_feature_names_out())
coefficients.round(3)

The numerical columns are standardized, so a coefficient is the change in the predicted value per standard deviation of its column, **with the other columns held fixed**. That condition matters: the coefficient of `AveRooms` is negative here although more rooms go with higher values on their own, because at a fixed income and a fixed number of bedrooms, more rooms mean something else. The large coefficients of `Latitude` and `Longitude` say that location matters a great deal once the other columns are fixed.

A prediction is the intercept plus one contribution per column, the prepared value times its coefficient. The intercept, 2.529, is the prediction for a district that is average in every numerical column and lies in the reference category of every label. The first validation district shows the sum at work; the cell lists the contributions larger than 0.05 in size.

In [ ]:
row = X_valid.iloc[[0]]
contributions = pd.Series(linear[0].transform(row)[0] * linear[-1].coef_,
                          index=linear[0].get_feature_names_out())
large = contributions[contributions.abs() > 0.05]
intercept_now = linear[-1].intercept_
totals = pd.Series({"intercept": intercept_now, "sum of all": intercept_now + contributions.sum(),
                    "prediction": linear.predict(row)[0]})
pd.concat([large, totals]).round(3)

The district lies inland, which lowers its prediction by 0.754, its income is below average, which lowers it by 0.363, and its latitude lowers it by another 0.406. Added to the intercept, all contributions give 1.256, exactly the model's prediction of about 126,000 dollars.

### Residual Diagnostics

A residual is the observed value minus the predicted value, and the pattern of the residuals shows where a model goes wrong:

- **residuals against predictions** should show a band of even width around 0. A curve means that the model misses a bend, and a fan that widens means that the errors grow with the prediction, and
- a **normal Q-Q plot** of the residuals, which Exercise 2 used to compare a sample with the normal distribution, should follow its line.

In [ ]:
predicted = linear.predict(X_valid)
residuals = y_valid - predicted
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(predicted, residuals, s=3, alpha=0.4, color=PALETTE[0])
axes[0].axhline(0, color=INK)
axes[0].set(xlabel="Predicted value", ylabel="Residual", title="Residuals against predictions")
(theoretical, ordered), (qq_slope, qq_intercept, _) = stats.probplot(residuals, dist="norm")
axes[1].scatter(theoretical, ordered, s=3, color=PALETTE[0])
axes[1].axline((0, qq_intercept), slope=qq_slope, color=PALETTE[1])
axes[1].set(xlabel="Normal quantile", ylabel="Ordered residual", title="Normal Q-Q plot of the residuals")
plt.show()

The residuals show two problems. A straight edge runs down the right of the first plot: these are the districts capped at 5.0, whose residual can only be 5.0 minus the prediction. Some predictions also fall below 0 or above 5, which no value can do. The Q-Q plot leaves its line at both ends, so large errors are more common than a normal error would allow. A straight plane through the predictors misses part of the pattern.

## 6. Polynomial Features

A straight line cannot follow a curve. `PolynomialFeatures` adds the powers of the columns, $x^2, x^3, \dots$, and the products of pairs of columns, such as $x_1 x_2$, as new columns, and a linear model on them fits a curved surface. The model is still linear in its coefficients, so least squares fits it as before.

For two columns $a = 2$ and $b = 3$, degree 2 gives $a$, $b$, $a^2$, $ab$, and $b^2$:

In [ ]:
toy_poly = PolynomialFeatures(degree=2, include_bias=False).fit(pd.DataFrame({"a": [2], "b": [3]}))
list(toy_poly.get_feature_names_out()), toy_poly.transform(pd.DataFrame({"a": [2], "b": [3]}))[0]

The product $ab$ is an **interaction**: it lets the effect of one column depend on the value of the other, for example the effect of income depending on the location. For the 8 numerical columns and degree 2, the new columns are a column of ones, the 8 columns themselves, their 8 squares, and the 28 products of two different columns, $\binom{8+2}{2} = 45$ in all.

In [ ]:
PolynomialFeatures(degree=2).fit(X_train[numeric].fillna(0)).n_output_features_

The degree is a hyperparameter, so it is chosen on the validation set. `include_bias=False` drops the column of ones, because the linear model fits its own intercept.

In [ ]:
rows = []
for degree in range(1, 5):
    model = make_pipeline(preparation, PolynomialFeatures(degree, include_bias=False), LinearRegression())
    model.fit(X_train, y_train)
    rows.append({"degree": degree, "columns": model[1].n_output_features_,
                 "training RMSE": root_mean_squared_error(y_train, model.predict(X_train)),
                 "validation RMSE": root_mean_squared_error(y_valid, model.predict(X_valid)),
                 "validation MAPE": mean_absolute_percentage_error(y_valid, model.predict(X_valid))})
degrees = pd.DataFrame(rows).set_index("degree")
degrees.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(degrees.index, degrees["training RMSE"], marker="o", label="training")
axes[0].plot(degrees.index, degrees["validation RMSE"], marker="o", label="validation")
axes[0].set(xlabel="Degree", ylabel="RMSE", title="RMSE by degree", xticks=degrees.index)
axes[0].legend()
axes[1].plot(degrees.index, degrees["validation MAPE"], marker="o", color=PALETTE[2])
axes[1].set(xlabel="Degree", ylabel="MAPE", title="Validation MAPE by degree", xticks=degrees.index)
plt.show()

The two metrics disagree. MAPE improves up to degree 3, while the validation RMSE of degree 3 jumps above that of a straight line, and degree 4 is worse still. A few districts with extreme averages, such as hundreds of occupants per household, receive wild predictions from the high powers. Those few errors barely move the average percentage error but dominate the squared error. Degree 2 has the lowest validation RMSE.

In [ ]:
best_poly = make_pipeline(preparation, PolynomialFeatures(2, include_bias=False), LinearRegression())
best_poly.fit(X_train, y_train)
results.loc["polynomial, degree 2"] = regression_scores(y_valid, best_poly.predict(X_valid))
results.round(3)

### Overfitting and Underfitting

The training error falls with every degree, because a higher degree can always fit the training rows at least as closely. This is **overfitting**: a model so flexible that it learns the noise of its training rows along with the pattern, and so predicts new rows worse. The opposite is **underfitting**, a model too rigid to follow the pattern, such as a straight line through a curve. The expected error of a model on new data splits into three parts:

- **bias**, the systematic error of a model that is too simple, which falls as flexibility grows,
- **variance**, how much the fit changes from one training sample to another, which grows with flexibility, and
- **noise**, the error that no model can remove.

The training error only ever falls, so it cannot find the balance between bias and variance. Only held-out rows reveal it.

The classic picture uses 15 noisy points from a smooth curve. A straight line misses the curve, which is underfitting. A cubic follows it. A polynomial of degree 12 passes close to every point and swings wildly between them, which is overfitting: it would predict new points from the same curve badly.

In [ ]:
x_toy = np.sort(np.random.default_rng(SEED).uniform(0, 1, 15))
y_toy = np.sin(2 * np.pi * x_toy) + np.random.default_rng(SEED + 1).normal(0, 0.25, 15)
x_line = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, degree in zip(axes, [1, 3, 12]):
    curve = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_toy.reshape(-1, 1), y_toy)
    ax.plot(x_line, np.sin(2 * np.pi * x_line), color=GRID, linewidth=3, label="true curve")
    ax.plot(x_line, curve.predict(x_line.reshape(-1, 1)), color=PALETTE[1], label=f"degree {degree}")
    ax.scatter(x_toy, y_toy, color=PALETTE[0], zorder=3)
    names = {1: "underfitting", 3: "a good fit", 12: "overfitting"}
    ax.set(ylim=(-2, 2), title=names[degree], xlabel="x")
    ax.legend(loc="lower left")
plt.show()

## 7. Cross-Validation

The validation set of 4,128 districts gives one score, and another split would give a slightly different one. **k-fold cross-validation** reuses the rows instead:

1. the rows are cut into k parts of equal size, called folds,
2. each fold in turn is held out, a model is fitted on the other k - 1 folds and scored on the held-out fold, and
3. the k scores are averaged.

Every row is held out exactly once, and the spread of the k scores shows how much a score moves with the split. With 5 folds, each model is fitted 5 times, each time on 80 % of the development rows, and the 5 held-out scores are averaged. The picture below marks, for every split, which rows are held out. `KFold` defines the folds, and `cross_val_score` fits and scores a model on each split. The cross-validation runs on the development rows, the training and validation rows together, so the test rows stay untouched.

In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=SEED)
[(len(fit_rows), len(held_out_rows)) for fit_rows, held_out_rows in folds.split(X_dev)]

In [ ]:
fold_rmse = -cross_val_score(best_poly, X_dev, y_dev, cv=folds, scoring="neg_root_mean_squared_error")
fold_rmse.round(3), round(fold_rmse.mean(), 3), round(fold_rmse.std(), 3)

Scikit-learn always maximizes a score, so an error is reported as a negative number, and the minus sign turns it back into an error. Two folds score like the validation set, and the other three score far worse, one of them with an RMSE of almost 10. Those folds hold a few districts with extreme averages, such as hundreds of occupants per household, and the squares and products of those values send their predictions far off. The single validation split hid how fragile the degree-2 model is, which is why a choice should rest on several splits. `cross_validate` returns several metrics at once, together with the training scores and the time of each fit.

In [ ]:
scoring = {"RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error", "R2": "r2"}
fold_scores = cross_validate(best_poly, X_dev, y_dev, cv=folds, scoring=scoring, return_train_score=True)
pd.DataFrame(fold_scores).round(3)

## 8. Regularization

Least squares picks the coefficients that make the training error as small as possible, and nothing stops it from using very large coefficients to shave a little off that error. With many columns, as polynomial features create, those coefficients swing wildly and the predictions with them, as the RMSE of degree 3 showed.

**Regularization** adds a penalty that grows with the size of the coefficients to the quantity being minimized:

$$\text{training error} + \lambda \times \text{size of the coefficients}$$

The model now balances fitting the training rows against keeping the coefficients small, and the **regularization strength** $\lambda \ge 0$ sets the balance. With $\lambda = 0$ the model is ordinary least squares, and with a very large $\lambda$ every coefficient is 0 and the model predicts the mean. The penalty trades a little bias for less variance. Three penalties are common:

- **Ridge regression** adds $\lambda \sum_j \beta_j^2$. It shrinks every coefficient towards 0 but sets none of them exactly to 0.
- **Lasso regression**, short for least absolute shrinkage and selection operator, adds $\lambda \sum_j |\beta_j|$. It sets some coefficients exactly to 0, which removes those columns from the model, so the lasso also selects variables.
- **Elastic net** adds a mix of the two, with the share of the lasso penalty set by a ratio $\rho$ between 0 and 1.

Scikit-learn calls $\lambda$ `alpha` and $\rho$ `l1_ratio`. The intercept is not penalized.

The effect of the strength is easy to see on the prepared columns. As `alpha` grows, ridge shrinks the coefficients towards 0. The validation RMSE barely moves at first, and it grows once the penalty is strong enough to remove real signal as well as noise.

In [ ]:
shrinking = {}
for strength in [0.1, 100, 1000, 10000]:
    ridge_now = make_pipeline(preparation, Ridge(alpha=strength)).fit(X_train, y_train)
    kept = ridge_now[-1].coef_[[0, 1, 6]].round(3)
    shrinking[strength] = dict(zip(["MedInc", "HouseAge", "Latitude"], kept))
    error = root_mean_squared_error(y_valid, ridge_now.predict(X_valid))
    shrinking[strength]["validation RMSE"] = round(error, 3)
pd.DataFrame(shrinking)

The latitude, which is correlated with the longitude, shrinks fastest, and the median income keeps more than half of its size even at `alpha=10000`. Not every coefficient shrinks steadily: `HouseAge` first grows a little while the correlated columns give way. A penalty decides how much size the coefficients may have in total, and it gives that size to the columns that reduce the error most.

### Scaling as a Precondition

A penalty treats every coefficient alike, but the size of a coefficient depends on the units of its column. The lasso with `alpha=1.0` on the eight raw columns shows what goes wrong.

In [ ]:
raw_lasso = make_pipeline(fill_only, Lasso(alpha=1.0)).fit(X_train, y_train)
pd.Series(raw_lasso[-1].coef_, index=numeric).round(4)

The lasso keeps only `MedInc` and `HouseAge` and removes the location entirely. The penalty is so strong at this `alpha` that it keeps only columns whose coefficient can stay large and useful, and the units decide which: a column measured in large numbers needs only a small coefficient. With standardized columns, every coefficient is measured per standard deviation, and the penalty compares like with like. From here on the penalty acts on standardized columns.

### Taming Degree 3

Degree 3 overfit badly. Its 559 columns are standardized, and `RidgeCV` and `LassoCV` choose the penalty by cross-validation on the training rows: they try every value in `alphas`, score each by cross-validation, and refit the best one on all the training rows. `max_iter` and `tol` limit the steps the lasso may take, because it has no closed form and is fitted step by step.

In [ ]:
def cubic(model: object) -> object:
    return make_pipeline(preparation, PolynomialFeatures(3, include_bias=False), StandardScaler(), model)


ridge = cubic(RidgeCV(alphas=np.logspace(-2, 4, 25))).fit(X_train, y_train)
lasso = cubic(LassoCV(alphas=np.logspace(-4, 0, 13), cv=folds, max_iter=5000, tol=1e-3))
lasso.fit(X_train, y_train)
elastic = cubic(ElasticNetCV(l1_ratio=[0.5, 0.8], alphas=np.logspace(-4, 0, 13), cv=folds,
                             max_iter=5000, tol=1e-3))
elastic.fit(X_train, y_train)
cubic_ols = cubic(LinearRegression()).fit(X_train, y_train)
for name, model in [("degree 3, least squares", cubic_ols), ("degree 3, ridge", ridge),
                    ("degree 3, lasso", lasso), ("degree 3, elastic net", elastic)]:
    results.loc[name] = regression_scores(y_valid, model.predict(X_valid))
results.round(3).tail(4)

In [ ]:
pd.DataFrame({"alpha": [ridge[-1].alpha_, lasso[-1].alpha_, elastic[-1].alpha_],
              "l1_ratio": [np.nan, 1, elastic[-1].l1_ratio_],
              "zero coefficients": [(m[-1].coef_ == 0).sum() for m in (ridge, lasso, elastic)]},
             index=["ridge", "lasso", "elastic net"])

The penalty turns the worst model into the best linear one. Least squares on the 559 columns has an $R^2$ near 0 on the validation rows, while ridge and the lasso bring it above 0.7. Of the 559 columns, 120 are 0 on every row: a district lies in one ocean category only, so a product of two different categories is always 0, and no model can use such a column. Ridge gives them coefficients at or near 0 and keeps every other column. The lasso sets 433 coefficients exactly to zero, so it keeps 126 powers and products and ignores the rest.

### Why the Lasso Sets Coefficients to Zero

Why does the lasso reach exactly 0 when ridge does not? For one coefficient $\beta$, the ridge penalty $\lambda\beta^2$ has a slope of $2\lambda\beta$, which vanishes as $\beta$ approaches 0, so ridge pushes a small coefficient only gently and never quite to 0. The lasso penalty $\lambda|\beta|$ has the same slope $\lambda$ however small $\beta$ is, so it keeps pushing until a coefficient that contributes little reaches 0 and stays there.

A penalty can also be read as a budget: the coefficients must stay inside a region, and the solution is the point of that region with the smallest training error. For two coefficients, the ridge region $\beta_1^2 + \beta_2^2 \le t$ is a disc, and the lasso region $|\beta_1| + |\beta_2| \le t$ is a diamond with corners on the axes. The contours of the training error spread outwards from the least-squares solution, and the first point at which they touch the region is the solution. A diamond is often touched at a corner, where one coefficient is exactly 0, while a disc has no corners.

In [ ]:
def training_error(b1: np.ndarray, b2: np.ndarray) -> np.ndarray:
    # An elliptical bowl around the least-squares solution (2, 0.4), as for two correlated columns.
    return (b1 - 2) ** 2 + 1.5 * (b2 - 0.4) ** 2 + 0.6 * (b1 - 2) * (b2 - 0.4)


b1_grid, b2_grid = np.meshgrid(np.linspace(-1.2, 3, 300), np.linspace(-1.2, 2, 300))
angle = np.linspace(0, 2 * np.pi, 2001)
circle = np.c_[np.cos(angle), np.sin(angle)]
shapes = {"lasso, |b1| + |b2| <= 1": circle / (np.abs(circle[:, :1]) + np.abs(circle[:, 1:])),
          "ridge, b1² + b2² <= 1": circle}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, (name, boundary) in zip(axes, shapes.items()):
    ax.contour(b1_grid, b2_grid, training_error(b1_grid, b2_grid), levels=12, colors=GRID)
    ax.fill(boundary[:, 0], boundary[:, 1], color=PALETTE[5], alpha=0.4)
    best = boundary[training_error(boundary[:, 0], boundary[:, 1]).argmin()]
    ax.scatter([2], [0.4], color=INK, label="least squares")
    ax.scatter([best[0]], [best[1]], color=PALETTE[1], zorder=3,
               label=f"penalized ({best[0]:.2f}, {best[1]:.2f})")
    ax.set(aspect="equal", title=name, xlabel="b1")
    ax.legend(loc="upper left")
axes[0].set_ylabel("b2")
plt.show()

The lasso solution sits on the corner where $\beta_2 = 0$, so the lasso drops the second column. The ridge solution keeps both coefficients and only shrinks them.

### The Regularization Path

The **path** of a penalty draws every coefficient against $\lambda$. `lasso_path` computes the lasso coefficients on the 13 prepared columns for a sequence of 100 penalties, from the smallest one that sets every coefficient to zero down to a thousandth of it.

In [ ]:
prepared = preparation.fit_transform(X_train)
names = preparation.get_feature_names_out()
path_alphas, path, _ = lasso_path(prepared, y_train - y_train.mean())
strong = {"numeric__MedInc": PALETTE[0], "numeric__Latitude": PALETTE[2],
          "numeric__Longitude": PALETTE[1]}
fig, ax = plt.subplots(figsize=(9, 3.8))
for row, name in zip(path, names):
    ax.plot(path_alphas, row, color=strong.get(name, GRID), linewidth=2 if name in strong else 1,
            label=name.split("__")[1] if name in strong else None)
ax.set(xscale="log", xlabel="alpha", ylabel="Coefficient", title="The lasso path")
ax.legend()
plt.show()

Read from the right, the path starts with every coefficient at zero, where the penalty is so strong that the model predicts the mean, and lets columns in as the penalty weakens. The median income enters first, because it carries most of the value. The latitude and the longitude enter late and then grow large and negative together, the same pattern that the coefficients of Section 5 showed: at a fixed income, districts further north or further east are worth less.

## 9. Choosing the Regression Model

The candidates are compared on their validation scores, which come from the training and validation rows only.

In [ ]:
results.sort_values("RMSE").round(3)

The elastic net and the lasso on the degree-3 features have the lowest validation RMSE, within 0.004 of each other. The choice is the lasso: it is as accurate as the elastic net, it has one hyperparameter fewer, and its many zero coefficients make it simpler to read. Exercise 6 shows how ensembles of trees improve on all of them.

The chosen model is refitted on the development rows, the training and validation rows together, because more rows give a better fit, and the test set is used once.

In [ ]:
final = lasso.fit(X_dev, y_dev)
test_baseline = DummyRegressor().fit(X_dev, y_dev)
pd.DataFrame({"baseline": regression_scores(y_test, test_baseline.predict(X_test)),
              "lasso, degree 3": regression_scores(y_test, final.predict(X_test))}).T.round(3)

On the test districts, the lasso removes about seven tenths of the squared deviation that the mean leaves, and its typical error is about 43,000 dollars, against about 90,000 for the baseline. The test RMSE is a little higher than the validation RMSE, as an honest estimate of new data usually is. Nothing may be changed after this step, because a model revised to improve its test score would make the test score optimistic.

### Summary

- Every model was judged on rows it never saw: a validation set or cross-validation chose, and the test set measured once.
- Every model was compared with a baseline that ignores the predictors, by metrics chosen for the cost of each error.
- Linear regression was fitted by a formula and by gradient descent, and its coefficients were read with the other columns held fixed.
- Residual plots showed where a straight plane misses, and polynomial features added curvature until they overfit.
- Regularization tamed the overfitting model, the penalty was chosen by cross-validation, the lasso selected its columns, and scaling came first.